<a href="https://colab.research.google.com/github/SchoolAccountLolXd/tuwaiq-assignment/blob/main/02_features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 2 — Feature Engineering
**ML Foundations Capstone | Titanic Dataset**

Starting from the cleaned dataset, we create features that are richer and more useful for analysis and modelling. Every new column includes a **WHY** explanation.

## 2.1 Imports & Load Clean Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100

# Load the cleaned output from Phase 1
df = pd.read_csv('../data/cleaned/titanic_clean.csv')

# Restore categorical dtypes (CSV does not preserve them)
df['survived'] = df['survived'].astype('category')
df['pclass']   = df['pclass'].astype('category')

print(f'Loaded: {df.shape}')
df.head()

## 2.2 One-Hot Encoding: `sex` and `embarked`

**WHY:** Models need numbers, not strings. One-hot encoding creates binary indicator columns without implying any false ordering between categories. `drop_first=True` removes one column per variable to avoid the dummy variable trap (perfect multicollinearity).

In [ ]:
# One-hot encode 'sex' and 'embarked'; drop_first avoids multicollinearity
df = pd.get_dummies(df, columns=['sex', 'embarked'], drop_first=True)

ohe_cols = [c for c in df.columns if c.startswith('sex_') or c.startswith('embarked_')]
print(f'New binary columns: {ohe_cols}')
df[ohe_cols].head(6)

## 2.3 Ordinal Encoding: `pclass`

**WHY:** Passenger class is ordered — 1st class is genuinely better than 3rd. We remap so **higher number = better class** (1st->3, 2nd->2, 3rd->1), making it directionally consistent with fare. This preserves the natural order that one-hot encoding would destroy.

In [ ]:
# Remap: original 1 (best) -> 3, original 2 -> 2, original 3 (lowest) -> 1
pclass_map = {1: 3, 2: 2, 3: 1}
df['pclass_ordinal'] = df['pclass'].astype(int).map(pclass_map)

print('Ordinal encoding mapping (pclass -> pclass_ordinal):')
print(df[['pclass', 'pclass_ordinal']].drop_duplicates().sort_values('pclass'))

## 2.4 StandardScaler: `age` and `fare`

**WHY:** Distance-based algorithms (k-NN, SVM) and gradient descent are sensitive to feature scale. StandardScaler transforms each column to mean=0, std=1 so age (range 0-80) and fare (range 0-500+) contribute equally to any model.

In [ ]:
scaler = StandardScaler()

# Store scaled values as new columns to preserve originals for EDA charts
df[['age_scaled', 'fare_scaled']] = scaler.fit_transform(df[['age', 'fare']])

print('Scaled columns — should have mean~0 and std~1:')
print(df[['age_scaled', 'fare_scaled']].describe().loc[['mean', 'std']].round(4))

## 2.5 Ratio Feature: `fare_per_class_unit`

**WHY:** A raw fare of 20 pounds means something very different for a 3rd-class ticket vs a 1st-class ticket. Dividing fare by the ordinal class score normalises spending relative to class level, revealing who over-paid or under-paid within their tier. Safe division guards against division by zero.

In [ ]:
# Safe division: np.where prevents any division-by-zero errors
df['fare_per_class_unit'] = np.where(
    df['pclass_ordinal'] > 0,
    df['fare'] / df['pclass_ordinal'],
    0
)

print('fare_per_class_unit stats:')
print(df['fare_per_class_unit'].describe().round(2))

## 2.6 Domain Feature: `family_size`

**WHY:** Historical accounts describe families struggling to evacuate together during the sinking. `family_size = sibsp + parch + 1` captures the full travel group size in one column. Solo travellers vs large family groups likely faced very different evacuation experiences.

In [ ]:
# sibsp = siblings/spouses aboard; parch = parents/children aboard
# Add 1 to count the passenger themselves
df['family_size'] = df['sibsp'] + df['parch'] + 1

print('family_size distribution:')
print(df['family_size'].value_counts().sort_index())

## 2.7 Interaction Feature: `class_fare_interaction`

**WHY:** The combination of high class AND high fare signals wealth more strongly than either feature alone. Multiplying `pclass_ordinal x fare_scaled` creates a composite luxury score. Interaction terms let linear models capture multiplicative (non-additive) relationships between features.

In [ ]:
df['class_fare_interaction'] = df['pclass_ordinal'] * df['fare_scaled']

print('class_fare_interaction stats:')
print(df['class_fare_interaction'].describe().round(3))

## 2.8 Log Transform: `fare`

**WHY:** `fare` is heavily right-skewed. A small number of very expensive first-class tickets drag the distribution far to the right. `np.log1p()` = log(1+x) compresses this tail into a much more symmetric shape, which benefits linear models and makes visualisations cleaner.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Before: raw fare distribution
axes[0].hist(df['fare'], bins=40, color='#4C72B0', edgecolor='white', alpha=0.85)
axes[0].set_title('Fare — Original (Skewed)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Fare (pounds)')
axes[0].set_ylabel('Count')

# Apply the log transform
df['fare_log'] = np.log1p(df['fare'])

# After: log-transformed distribution
axes[1].hist(df['fare_log'], bins=40, color='#55A868', edgecolor='white', alpha=0.85)
axes[1].set_title('Fare — After log1p Transform', fontsize=13, fontweight='bold')
axes[1].set_xlabel('log(1 + Fare)')
axes[1].set_ylabel('Count')

plt.suptitle('Log Transformation Reduces Fare Skewness', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/cleaned/log_transform.png', bbox_inches='tight')
plt.show()

print(f'Skewness BEFORE: {df["fare"].skew():.3f}')
print(f'Skewness AFTER:  {df["fare_log"].skew():.3f}')

## 2.9 Bin `age` into Life-Stage Groups

**WHY:** Treating age as a continuous number assigns equal importance to every year gap. Grouping into life stages (Child, Youth, Adult, Middle-Aged, Senior) lets us analyse survival patterns at a meaningful level — a key question for the Titanic, where the evacuation order was 'women and children first'.

In [ ]:
age_bins   = [0, 12, 18, 35, 60, float('inf')]
age_labels = ['Child', 'Youth', 'Adult', 'Middle-Aged', 'Senior']

df['age_group'] = pd.cut(df['age'], bins=age_bins, labels=age_labels, right=True)

print('Age group distribution:')
print(df['age_group'].value_counts().sort_index())

## 2.10 Remove Redundant Features (correlation > 0.99)

**WHY:** Features with near-perfect correlation (r > 0.99) carry essentially identical information. We use a conservative 0.99 threshold here to only drop features that are exact linear transforms of another (such as `age_scaled` vs `age`), while preserving meaningful engineered features like `family_size` and `class_fare_interaction` that have slightly lower correlations with their source columns.

In [ ]:
# Select numeric columns for correlation analysis
numeric_df  = df.select_dtypes(include=[np.number])
corr_matrix = numeric_df.corr().abs()

# Use upper triangle only — avoids counting each pair twice
upper   = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if any(upper[col] > 0.99)]

print(f'Columns to drop (r > 0.99 with another feature): {to_drop}')

if to_drop:
    df = df.drop(columns=to_drop)
    print(f'Dropped: {to_drop}')
else:
    print('No redundant pairs found — all features retained.')

print(f'\nFinal dataset shape: {df.shape}')

## 2.11 Save Feature-Engineered Dataset

In [ ]:
df.to_csv('../data/cleaned/titanic_features.csv', index=False)
print(f'Saved -> data/cleaned/titanic_features.csv')
print(f'Final shape: {df.shape}')
df.head()